In [1]:
import pandas as pd
import numpy as np
import random

# Pour le GNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import  StandardScaler

# Pour faire un "value counts"
from collections import Counter

# Convertir des str en dict même si eval le fait déjà
import ast 
# Retirer les warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# df_users = pd.read_parquet("user1.parquet")
# df_rest = pd.read_json("business1.json")
# df_review = pd.read_parquet("review1.parquet")

In [ ]:
# df_rest_b = df_rest.drop(columns=['name', 'address', 'city', 'state', 'postal_code',
#        'latitude', 'longitude', 'is_open', 'HairSpecializesIn', 'AcceptsInsurance'])



In [ ]:
# df_users_b = df_users.drop(columns=['compliment_hot',
#        'compliment_more', 'compliment_profile', 'compliment_cute',
#        'compliment_list', 'compliment_note', 'compliment_plain',
#        'compliment_cool', 'compliment_funny', 'compliment_writer',
#        'compliment_photos', 'friends', 'name', 'fans', 'useful', 'funny',
#        'cool', 'elite'])

In [ ]:
# df_review_b = df_review.drop(columns=['useful', 'funny',
#        'cool', 'date', 'text'])



---

In [2]:
# E1:

# Charger les datasets
users_df = pd.read_parquet('user1.parquet')
reviews_df = pd.read_parquet('review1.parquet')
restaurants_df = pd.read_json("business1.json")

reviews_df = reviews_df.drop(columns=['useful', 'funny',
       'cool', 'date', 'text'])

restaurants_df = restaurants_df.drop(columns=['name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'is_open', 'HairSpecializesIn', 'AcceptsInsurance'])

users_df = users_df.drop(columns=['compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos', 'friends', 'name', 'fans', 'useful', 'funny',
       'cool', 'elite', 'yelping_since'])

# E2: Subset

# Sélectionner les users les plus actifs
top_users = users_df.nlargest(10000, 'review_count')['user_id'].values

# Filtrer les reviews de ces users
subset_reviews = reviews_df[reviews_df['user_id'].isin(top_users)].copy()

# Sélectionner les restaurants les plus reviewés parmi ces reviews
top_restaurants = subset_reviews['business_id'].value_counts().head(5000).index.values

# Filtrer les reviews pour ne garder que ces restaurants
subset_reviews = subset_reviews[subset_reviews['business_id'].isin(top_restaurants)].copy()

# Filtrer les dataframes
subset_users = users_df[users_df['user_id'].isin(subset_reviews['user_id'].unique())].copy()
subset_restaurants = restaurants_df[restaurants_df['business_id'].isin(top_restaurants)].copy()

# E3:preprocessing

subset_users['review_count_log'] = np.log1p(subset_users['review_count'])
subset_users['average_stars_norm'] = subset_users['average_stars'] / 5.0

user_features_cols = ['review_count_log', 'average_stars_norm']


col_to_dict = ["BusinessParking", "Ambience", "GoodForMeal"]
for col in col_to_dict:
    subset_restaurants[col] = subset_restaurants[col].apply(lambda x: ast.literal_eval(x))

# Features numériques de base
subset_restaurants['stars_norm'] = subset_restaurants['stars'] / 5.0
subset_restaurants['review_count_log'] = np.log1p(subset_restaurants['review_count'])

# Price Range
subset_restaurants['RestaurantsPriceRange2'] = pd.to_numeric(subset_restaurants['RestaurantsPriceRange2'], errors='coerce').fillna(2.0) 
subset_restaurants['price_norm'] = subset_restaurants['RestaurantsPriceRange2'] / 4.0


def top_k_categories(k: int):
    # Top k catégories
    all_categories = []
    for cats in subset_restaurants['categories'].dropna():
        if isinstance(cats, str):
            all_categories.extend([c.strip() for c in cats.split(',')])
    cats = [cat for cat, _ in Counter(all_categories).most_common(k)]
    return cats 

top_20_categories = top_k_categories(20)
for cat in top_20_categories:
    subset_restaurants[f'cat_{cat}'] = subset_restaurants['categories'].apply(lambda x: 1.0 if isinstance(x, str) and cat in x else 0.0)

# Attributs booléens
boolean_attrs = [
    'RestaurantsDelivery', 'OutdoorSeating', 'BusinessAcceptsCreditCards',
    'BikeParking', 'RestaurantsTakeOut', 'GoodForKids', 'Caters',
    'RestaurantsReservations', 'RestaurantsTableService', 'RestaurantsGoodForGroups',
    'WheelchairAccessible', 'HasTV', 'HappyHour', 'DriveThru',
    'BusinessAcceptsBitcoin', 'DogsAllowed', 'Open24Hours'
]

def parse_boolean(val):
    if pd.isna(val) or val is None or val == 'None':
        return 0.0
    if isinstance(val, bool):
        return 1.0 if val else 0.0
    if isinstance(val, str):
        val_lower = val.lower()
        if val_lower in ['true', '1', 'yes']:
            return 1.0
        elif val_lower in ['false', '0', 'no']:
            return 0.0
    return 0.0

for attr in boolean_attrs:
    if attr in subset_restaurants.columns:
        subset_restaurants[f'{attr}_bool'] = subset_restaurants[attr].apply(parse_boolean)

# Attributs catégoriels
categorical_attrs = {
    'WiFi': ['free', 'no', 'paid'],
    'Alcohol': ['full_bar', 'beer_and_wine', 'none'],
    'NoiseLevel': ['quiet', 'average', 'loud', 'very_loud'],
    'RestaurantsAttire': ['casual', 'dressy', 'formal'],
    'Smoking': ['no', 'yes', 'outdoor'],
    "BusinessParking": ['garage', 'lot', 'street', 'valet', 'validated'],
    "Ambience": ['casual', 'classy', 'divey', 'hipster', 'intimate', 'romantic', 'touristy', 'trendy','upscale'],
    "GoodForMeal": ['lunch', 'brunch', 'dessert', 'dinner', 'latenight', 'breakfast']
}

for attr, values in categorical_attrs.items():
    if attr in subset_restaurants.columns:
        for val in values:
            subset_restaurants[f'{attr}_{val}'] = subset_restaurants[attr].apply(lambda x: 1.0 if isinstance(x, str) and val in x.lower() and x!="None" else 0.0)

# Compiler toutes les features restaurants
restaurant_features_cols = (
    ['stars_norm', 'review_count_log', 'price_norm'] +
    [f'cat_{cat}' for cat in top_20_categories] +
    [f'{attr}_bool' for attr in boolean_attrs if attr in subset_restaurants.columns] +
    [f'{attr}_{val}' for attr, values in categorical_attrs.items() 
     for val in values if attr in subset_restaurants.columns]
)

subset_restaurants_fillna = subset_restaurants[restaurant_features_cols].fillna(0)
subset_users_fillna = subset_users[user_features_cols].fillna(0)

In [3]:
# E4: Création du Graphe

# Créer les mappings
user_to_idx = {uid: i for i, uid in enumerate(subset_users['user_id'])}
num_users = len(user_to_idx)
restaurant_to_idx = {rid: i+ num_users for i, rid in enumerate(subset_restaurants['business_id'])}
restaurant_idx_id = {i+ num_users: rid  for i, rid in enumerate(subset_restaurants['business_id'])}


num_restaurants = len(restaurant_to_idx)


# Normaliser les features
scaler_user = StandardScaler()
scaler_restaurant = StandardScaler()

user_features = scaler_user.fit_transform(subset_users_fillna)
restaurant_features = scaler_restaurant.fit_transform(subset_restaurants_fillna)

# Padding pour avoir la même dimension
max_dim = max(user_features.shape[1], restaurant_features.shape[1])

user_features_padded = np.pad(
    user_features,
    ((0, 0), (0, max_dim - user_features.shape[1])),
    mode='constant'
)
restaurant_features_padded = np.pad(
    restaurant_features,
    ((0, 0), (0, max_dim - restaurant_features.shape[1])),
    mode='constant'
)

# Combiner
all_features = np.vstack([user_features_padded, restaurant_features_padded])
x = torch.FloatTensor(all_features)

edge_list = []
edge_labels = []

# Likes 
likes = subset_reviews[subset_reviews['stars'] >= 3]
for _, row in likes.iterrows():
    if row['user_id'] in user_to_idx and row['business_id'] in restaurant_to_idx:
        user_idx = user_to_idx[row['user_id']]
        restaurant_idx = restaurant_to_idx[row['business_id']]
        
        edge_list.append([user_idx, restaurant_idx])
        edge_list.append([restaurant_idx, user_idx])
        edge_labels.extend([1.0, 1.0])

# Dislikes
dislikes = subset_reviews[subset_reviews['stars'] <= 2]
for _, row in dislikes.iterrows():
    if row['user_id'] in user_to_idx and row['business_id'] in restaurant_to_idx:
        user_idx = user_to_idx[row['user_id']]
        restaurant_idx = restaurant_to_idx[row['business_id']]
        
        edge_list.append([user_idx, restaurant_idx])
        edge_list.append([restaurant_idx, user_idx])
        edge_labels.extend([0.0, 0.0])

edge_index = torch.LongTensor(edge_list).t().contiguous()
edge_label = torch.FloatTensor(edge_labels)


print(f"Edges: {edge_index.shape[1]:,}")
print(f"Likes: {(edge_label == 1.0).sum().item():,}")
print(f"Dislikes: {(edge_label == 0.0).sum().item():,}")

graph_data = Data(x=x, edge_index=edge_index, edge_label=edge_label)


Edges: 337,124
Likes: 310,604
Dislikes: 26,520


In [4]:
# E5: GNN

class RestaurantGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.predictor = nn.Sequential(
            nn.Linear(out_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, 1)
        )
        
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x
    
    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return self.predictor(torch.cat([src, dst], dim=1)).squeeze()
    
    def forward(self, x, edge_index, edge_label_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_label_index)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RestaurantGNN(
    in_channels=max_dim,
    hidden_channels=128,
    out_channels=64
).to(device)



In [ ]:
# E6: Training

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.BCEWithLogitsLoss()

data = graph_data.to(device)

num_epochs = 50

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Negative sampling
    neg_edge_index = negative_sampling(
        edge_index=data.edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=data.edge_index.size(1) // 2
    )
    
    # Combiner edges positifs et négatifs
    edge_label_index = torch.cat([data.edge_index, neg_edge_index], dim=1)
    edge_label = torch.cat([
        data.edge_label,
        torch.zeros(neg_edge_index.size(1))
    ], dim=0).to(device)
    
    # Forward pass
    out = model(data.x, data.edge_index, edge_label_index)
    loss = criterion(out, edge_label)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Metrics
    if epoch % 5 == 0:
        with torch.no_grad():
            pred = out.cpu().numpy()
            true = edge_label.cpu().numpy()
            auc = roc_auc_score(true, pred) if len(np.unique(true)) > 1 else 0.5
            acc = ((pred > 0.5) == true).mean()
            
            print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | AUC: {auc:.4f} | Acc: {acc:.4f}")

# E7: recommandation

def get_top_k_recommendations(user_id, k=20, verbose=True):
    """
    Obtient les top K recommandations pour un user
    
    Args:
        user_id: ID du user
        k: Nombre de recommandations
        verbose: Afficher les détails
    
    Returns:
        List of (restaurant_id, score, restaurant_info)
    """
    if user_id not in user_to_idx:
        print(f"User {user_id} non trouvé dans le subset")
        return []
    
    model.eval()
    with torch.no_grad():
        # Encoder le graphe
        z = model.encode(data.x, data.edge_index)
        
        user_idx = user_to_idx[user_id]
        
        # Calculer les scores pour tous les restaurants
        scores = []
        for restaurant_id, restaurant_idx in restaurant_to_idx.items():
            
            edge_label_index = torch.tensor(
                [[user_idx], [restaurant_idx]], 
                device=device
            )
            score = model.decode(z, edge_label_index).item()
            scores.append((restaurant_id, score))
        
        # Trier par score décroissant
        scores.sort(key=lambda x: x[1], reverse=True)
        top_k = scores[:k]
        
        # Enrichir avec les infos restaurants
        recommendations = []
        for restaurant_id, score in top_k:
            restaurant_info = subset_restaurants[subset_restaurants['business_id'] == restaurant_id].iloc[0].to_dict()
            recommendations.append((restaurant_id, score, restaurant_info))
        
        if verbose:
            print(f"Top {k} recommandations pour user {user_id}:")
            for i, (rid, score, info) in enumerate(recommendations, 1):
                print(f"{i}. Score: {score:.4f} | Stars: {info['stars']:.1f} | {info.get('categories', 'N/A')}")
        
        return recommendations

# Test avec un user aléatoire
test_user = subset_users.iloc[0]['user_id']
recs = get_top_k_recommendations(test_user, k=20)


Epoch 000 | Loss: 0.6974 | AUC: 0.4642 | Acc: 0.3858
Epoch 005 | Loss: 0.6532 | AUC: 0.6929 | Acc: 0.4379
Epoch 010 | Loss: 0.5480 | AUC: 0.7825 | Acc: 0.7079
Epoch 015 | Loss: 0.4529 | AUC: 0.8183 | Acc: 0.7950
Epoch 020 | Loss: 0.3904 | AUC: 0.8624 | Acc: 0.8374
Epoch 025 | Loss: 0.3694 | AUC: 0.8725 | Acc: 0.8465
Epoch 030 | Loss: 0.3600 | AUC: 0.8847 | Acc: 0.8419
Epoch 035 | Loss: 0.3465 | AUC: 0.8935 | Acc: 0.8518
Epoch 040 | Loss: 0.3412 | AUC: 0.8970 | Acc: 0.8553
Epoch 045 | Loss: 0.3348 | AUC: 0.9017 | Acc: 0.8577
Top 20 recommandations pour user qVc8ODYU5SZjKXVBgXdI7w:
1. Score: 3.0329 | Stars: 4.5 | Candy Stores, Shopping, Department Stores, Fast Fo
2. Score: 2.7962 | Stars: 4.5 | American (Traditional), Chicken Shop, Southern, Re
3. Score: 2.4213 | Stars: 4.5 | French, Restaurants, Cocktail Bars, Nightlife, Ame
4. Score: 2.3219 | Stars: 4.5 | Food, Restaurants, Caterers, Smokehouse, Barbeque,
5. Score: 2.2073 | Stars: 4.5 | Shopping, Butcher, Restaurants, Sandwiches, Food,

---

In [ ]:
def add_edge(user_id, rest_id_list, preference_list):
    edge = graph_data.edge_index.T.tolist()
    label = graph_data.edge_label.cpu().tolist()
    
    exist_edges = list(map(lambda x: (restaurant_idx_id[x[0]], x[1]), [(elt[1],l) for elt, l in zip(edge, label) if elt[0] == user_to_idx[user_id]]))
    exist_res = [r[0] for r in exist_edges]
    exist_label = [r[1] for r in exist_edges]

    if user_id not in user_to_idx:
        print("user is not in user_to_idx")
        return False
    
    if not (rest_id_list and all([True if rest in restaurant_to_idx else False for rest in rest_id_list])):
        print("restaurant not in restaurant_to_idx")
        return False
    
    if set(preference_list) not in [{1, 0}, {1}, {0}]:
        return False
    
    if len(preference_list) != len(rest_id_list):
        return False
  
    rmv = []
    # Change like to dislike
    for i, (res, pref) in enumerate(zip(rest_id_list, preference_list)):
        if res in exist_res:
            if pref == exist_label[exist_res.index(res)]:
                return False
            else:
                index = graph_data.edge_index.T.cpu().tolist().index([user_to_idx[user_id], restaurant_to_idx[res]])
                if pref == 0:
                    graph_data.edge_label[index] = 1
                else: 
                    graph_data.edge_label[index] = 0
                rmv.append(i)

    
    rest_id_list = [rest_id_list[i] for i in range(len(rest_id_list)) if i not in rmv]
    preference_list = [preference_list[i] for i in range(len(preference_list)) if i not in rmv]

    
    edge_list = []
    edge_labels = []

    user_idx = user_to_idx[user_id]
    for rest, pref in zip(rest_id_list, preference_list):
        restaurant_idx = restaurant_to_idx[rest]
        edge_list.append([user_idx, restaurant_idx])
        edge_list.append([restaurant_idx, user_idx])

        if pref == 1:
            edge_labels.extend([1.0, 1.0])
        else:
            edge_labels.extend([0.0, 0.0])

    new_edge_index = torch.LongTensor(edge_list).t().contiguous()
    new_edge_label = torch.FloatTensor(edge_labels)
    
    graph_data.edge_index = torch.cat([graph_data.edge_index, new_edge_index], dim=1)
    graph_data.edge_label = torch.cat([graph_data.edge_label, new_edge_label])
    
    return True


In [ ]:
# Ceci est un TEST pour modifier le label d'une arête

# user_id = 'si99SgWBLfk0WX3dZJhWlA'
# rest_id = '9OG5YkX1g2GReZM0AskizA'
# index = graph_data.edge_index.T.cpu().tolist().index([user_to_idx[user_id], restaurant_to_idx[rest_id]])
# graph_data.edge_label[index] = 1


In [ ]:
# Ceci est un TEST pour tester la fonction add_edge

# add_edge(user_id="R[x{=dx1dOy<{OFaRG4EAh", rest_id_list=["9OG5YkX1g2GReZM0AskizA", "YNjyv0gfOr2g8lbmUpTnKg"], preference_list=[1, 0])

True

In [45]:
user_to_idx[id]

14897

In [46]:
# Ceci est un TEST pour permettre de voir les relations existantes pour un restaurant donné

edge = graph_data.edge_index.T.cpu().tolist()
edges = []
for elt in edge:
    if elt[0] == 14897:
        edges.append(elt)
print(edges)

[[14897, 14547], [14897, 10574], [14897, 11646], [14897, 11734], [14897, 13679], [14897, 10744]]


In [47]:
lst = [14547, 10574,  11646]
for i in lst:
    display(subset_restaurants[subset_restaurants["business_id"] == restaurant_idx_id[i]])
    print()

,business_id,stars,review_count,categories,hours,RestaurantsDelivery,OutdoorSeating,BusinessAcceptsCreditCards,BusinessParking,BikeParking,...,Ambience_romantic,Ambience_touristy,Ambience_trendy,Ambience_upscale,GoodForMeal_lunch,GoodForMeal_brunch,GoodForMeal_dessert,GoodForMeal_dinner,GoodForMeal_latenight,GoodForMeal_breakfast
32583,r0g6naqWCKdKcTWuYibtNQ,4.5,327,"Japanese, Asian Fusion, Restaurants, Sushi Bars","{'Monday': '16:0-21:0', 'Tuesday': '16:0-22:0'...",None,True,True,"{'garage': True, 'street': True, 'validated': ...",True,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,business_id,stars,review_count,categories,hours,RestaurantsDelivery,OutdoorSeating,BusinessAcceptsCreditCards,BusinessParking,BikeParking,...,Ambience_romantic,Ambience_touristy,Ambience_trendy,Ambience_upscale,GoodForMeal_lunch,GoodForMeal_brunch,GoodForMeal_dessert,GoodForMeal_dinner,GoodForMeal_latenight,GoodForMeal_breakfast
4582,DkePId_VZgMx7skQ7aqMGw,4.0,229,"Sushi Bars, Restaurants, Thai, Japanese","{'Monday': '0:0-0:0', 'Wednesday': '16:30-21:0...",True,None,True,"{'garage': False, 'street': False, 'validated'...",True,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,business_id,stars,review_count,categories,hours,RestaurantsDelivery,OutdoorSeating,BusinessAcceptsCreditCards,BusinessParking,BikeParking,...,Ambience_romantic,Ambience_touristy,Ambience_trendy,Ambience_upscale,GoodForMeal_lunch,GoodForMeal_brunch,GoodForMeal_dessert,GoodForMeal_dinner,GoodForMeal_latenight,GoodForMeal_breakfast
12052,EAPAK_XoJMXtUADVkB3xoA,3.5,292,"Japanese, Ramen, Restaurants","{'Monday': '16:0-20:0', 'Wednesday': '16:0-20:...",True,False,True,"{'garage': False, 'street': True, 'validated':...",True,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Ceci est un TEST pour voir les restaurants likés ou dislikés par un user

# subset_reviews[subset_reviews["user_id"] == "si99SgWBLfk0WX3dZJhWlA"].review_id.tolist()

In [ ]:
def get_propositions(idx):

    for i in idx:
        if len(i) != 0:
            print(subset_restaurants[subset_restaurants.index.isin(i) & (subset_restaurants["stars"]>=3)].sort_values(by = "stars", ascending=False).sample(5 if len(i)>=5 else len(i)))


def select_categories(user_id):
    print("What type of food do you prefer ?")
    cat = top_k_categories(30)
    print(cat)
    # provisoire car on n'a pas d'interface encore 
    # Pas de blindage ici
    ipt = str(input("Input: ")).split()
    idx = []
    for i in ipt:
        idx.append(subset_restaurants[subset_restaurants["categories"].str.contains(i)].index.tolist())
    
    get_propositions(idx)

    print("\n# Like/ Dislike:")
    ipt_rest = str(input("Restaurant ID's:")).split()
    ipt_labels = list(map(int, str(input("Lables:")).split()))
    return add_edge(user_id, ipt_rest, ipt_labels)


    


In [18]:
def generate_user_id():
    user_id = "".join([chr(random.randint(48, 125)) for _ in range(22)])
    if user_id in user_to_idx:
        generate_user_id()
    else:
        return user_id
    

def add_user():
    new_user_features_raw = np.array([[0, 0.7]])

    # Normalisation
    new_user_features = scaler_user.transform(new_user_features_raw)

    new_user_features_padded = np.pad(
        new_user_features,
        ((0, 0), (0, max_dim - new_user_features.shape[1])),
        mode='constant'
    )

    user_id = generate_user_id()
    user_to_idx[user_id] = len(restaurant_to_idx)+len(user_to_idx)

    graph_data.x = torch.cat([
            graph_data.x,
            torch.FloatTensor(new_user_features_padded)
        ], dim=0)
    
    return user_id

In [ ]:
def cold_start():
    u_id = add_user()
    val = select_categories(u_id)
    if val:
        print("Success !")
    else:
        print("Error...")
    
    return u_id 


id = cold_start()

recs = get_top_k_recommendations(id, k=10)
    

What type of food do you prefer ?
['Restaurants', 'Food', 'Nightlife', 'Bars', 'American (Traditional)', 'American (New)', 'Breakfast & Brunch', 'Sandwiches', 'Seafood', 'Event Planning & Services', 'Burgers', 'Italian', 'Pizza', 'Coffee & Tea', 'Mexican', 'Cocktail Bars', 'Salad', 'Cafes', 'Desserts', 'Pubs', 'Steakhouses', 'Sushi Bars', 'Bakeries', 'Specialty Food', 'Beer', 'Wine & Spirits', 'Arts & Entertainment', 'Japanese', 'Caterers', 'Wine Bars']


                  business_id  stars  review_count  \
32583  r0g6naqWCKdKcTWuYibtNQ    4.5           327   
14467  yE9JBfdRUlDsKaAIn0yOAA    4.5           250   
4582   DkePId_VZgMx7skQ7aqMGw    4.0           229   
12052  EAPAK_XoJMXtUADVkB3xoA    3.5           292   
12695  DMxgL0TdpR9HCD4P1CdxdQ    3.0           144   

                                              categories  \
32583    Japanese, Asian Fusion, Restaurants, Sushi Bars   
14467                  Japanese, Restaurants, Sushi Bars   
4582             Sushi Bars, Restaurants, Thai, Japanese   
12052                       Japanese, Ramen, Restaurants   
12695  Restaurants, Japanese, Chinese, Gluten-Free, K...   

                                                   hours RestaurantsDelivery  \
32583  {'Monday': '16:0-21:0', 'Tuesday': '16:0-22:0'...                None   
14467  {'Monday': '0:0-0:0', 'Tuesday': '17:0-21:0', ...                True   
4582   {'Monday': '0:0-0:0', 'Wednesday': '16:30-21:0...                Tr

In [56]:
recs = get_top_k_recommendations(id, k=10)


Top 10 recommandations pour user bwnjY;nxp2TJ6J`Y}_{jcq:
1. Score: -2.0356 | Stars: 4.0 | Restaurants, Mexican
2. Score: -2.2493 | Stars: 4.0 | Southern, Barbeque, Restaurants, Breakfast & Brunch, Sandwiches, Canadian (New)
3. Score: -2.2743 | Stars: 4.5 | Patisserie/Cake Shop, Food, Restaurants, Bakeries, Cafes, French
4. Score: -2.2965 | Stars: 3.0 | Arts & Entertainment, American (Traditional), Restaurants, Active Life, Hotels, Event Planning & Services, Hotels & Travel, Casinos, Nightlife, Music Venues, Escape Games
5. Score: -2.7781 | Stars: 4.5 | Restaurants, Vegan, Indonesian, Vegetarian
6. Score: -2.8054 | Stars: 3.0 | Nightlife, Bars, Pubs, Mexican, Restaurants
7. Score: -2.8492 | Stars: 4.0 | Restaurants, Southern, Cajun/Creole
8. Score: -2.8561 | Stars: 4.0 | Cafes, Vegetarian, Food, Bakeries, Restaurants, Vegan
9. Score: -2.8689 | Stars: 4.5 | Tapas Bars, Japanese, Restaurants, Korean
10. Score: -2.9155 | Stars: 4.5 | Canadian (New), Restaurants
